In [ ]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import sys

sys.path.insert(0, "/home/joshua/PhD_year_1/jaxsp/Adding_stellar_masses")


import bessel_test as BT


import analytic_test_gaunt as ATG

import Analytic_test_CC_speed as ATCC

import jaxsp as jsp

import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
import jax.numpy as jnp

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


# Using analytic method - either Gaunt coeffs or using s2fft

## 1) Static, spherically symmetric profile using ias15 - 100 timesteps

In [ ]:

import importlib
importlib.reload(ATCC)


static = True
frozen = False
SphHT = True
time_dep = False
integrator = 'leapfrog'
plot = False
dt_override = 30

frozen_test_analytic = ATCC.StellarSimTDep(m22 = 1, r_half = 0.19, no_of_particles = 5, no_time_steps = 50, total_evolve_time = 10, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, static = static, frozen = frozen, SphHT = SphHT, 
                               integrator = integrator, plot = plot, dt_override = dt_override, time_dep = time_dep)

In [ ]:

frozen_test_analytic.run_simulation()

In [ ]:
# Convert to numpy for plotting/analysis
import numpy as np

positions_all = np.array([p.positions_xyz for p in frozen_test_analytic.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in frozen_test_analytic.particles])  # (N_particles, N_steps+1)
all_vels_cart = np.array([[np.array(v) for v in p.velocities_cart] for p in frozen_test_analytic.particles])
kinetic_energy_all = np.array([p.kinetic_energy for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1, 3)

In [ ]:

print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib 
matplotlib.rcParams['animation.embed_limit'] = 200  
from IPython.display import HTML

N, T, _ = positions_all.shape
dt_Gyr = frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

# Fix axis limits so the view doesn't jump
xyz_min = positions_all.reshape(-1, 3).min(axis=0) * frozen_test_analytic.u.to_Kpc
xyz_max = positions_all.reshape(-1, 3).max(axis=0) * frozen_test_analytic.u.to_Kpc
ax.set_xlim(xyz_min[0], xyz_max[0])
ax.set_ylim(xyz_min[1], xyz_max[1])
ax.set_zlim(xyz_min[2], xyz_max[2])
ax.set_xlabel('X [kpc]'); ax.set_ylabel('Y [kpc]'); ax.set_zlabel('Z [kpc]')

# One trail line + one head point per particle
trails = [ax.plot([], [], [], lw=0.8)[0] for _ in range(N)]
heads  = [ax.plot([], [], [], 'o', ms=3)[0] for _ in range(N)]

# Time readout
time_text = ax.text2D(0.02, 0.98, '', transform=ax.transAxes,
                      verticalalignment='top', fontsize=11,
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

trail_len = 50  # frames of history to draw; set to T for full trail

def update(frame):
    start = max(0, frame - trail_len)
    for i in range(N):
        seg = positions_all[i, start:frame+1] * frozen_test_analytic.u.to_Kpc
        trails[i].set_data(seg[:, 0], seg[:, 1])
        trails[i].set_3d_properties(seg[:, 2])
        p = positions_all[i, frame] * frozen_test_analytic.u.to_Kpc
        heads[i].set_data([p[0]], [p[1]])
        heads[i].set_3d_properties([p[2]])
    time_text.set_text(f't = {frame * dt_Gyr:.3f} Gyr')
    return trails + heads + [time_text]

anim = FuncAnimation(fig, update, frames=T, interval=40, blit=False)
HTML(anim.to_jshtml())   # inline player; or anim.save('orbits.mp4', fps=25)

In [ ]:
time_step2 = frozen_test_analytic.time_step
stellar_v_disp2 = np.sqrt(
    np.std(all_vels_cart[:, :, 0], axis=0)**2 +
    np.std(all_vels_cart[:, :, 1], axis=0)**2 +
    np.std(all_vels_cart[:, :, 2], axis=0)**2
)
average_r2 = np.mean(r_all, axis=0)  # Average over particles


x = np.linspace(0, time_step2, len(stellar_v_disp2))

plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, stellar_v_disp2 * frozen_test_analytic.u.to_kms)
plt.yscale('log')
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, average_r2 * frozen_test_analytic.u.to_Kpc, label='Average Particle Radius')
for particle in range(r_all.shape[0]):
    plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, r_all[particle] * frozen_test_analytic.u.to_Kpc, alpha = 0.1, color='gray')
plt.axhline(frozen_test_analytic.r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')
plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')

# Timescale diagnostics
v0 = np.sqrt(2 * kinetic_energy_all[:, 0])
mean_T_orb = float(np.mean(2 * np.pi * r_all[:, 0] / v0) * frozen_test_analytic.u.to_Gyr)

lambda_db_kpc = 19.15 / (frozen_test_analytic.m22 * v0 * frozen_test_analytic.u.to_kms)
T_c = lambda_db_kpc / (v0 * frozen_test_analytic.u.to_Kpc) * frozen_test_analytic.u.to_Gyr

#plt.axhline(np.mean(lambda_db_kpc), color='g', linestyle='--', label='$\\lambda_{{\\rm db}}$')


E = np.array(frozen_test_analytic.eigen_energies)
freq_diff = np.abs(E[:, None] - E[None, :])
T_beat = (2 * np.pi / freq_diff) * frozen_test_analytic.u.to_Gyr
min_T_beat = np.min(T_beat[np.isfinite(T_beat)])
max_T_beat = np.max(T_beat[np.isfinite(T_beat)])

dt_Gyr = frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr

info = (
    f"$T_{{\\rm orb}}$ (mean) = {mean_T_orb:.3f} Gyr\n"
    f"$T_{{\\rm c}}$ = {float(np.mean(T_c)):.3f} Gyr\n"
    f"Beat time band: [{min_T_beat:.3f}, {max_T_beat:.3f}] Gyr\n"
    f"$\\Delta t$ = {dt_Gyr:.4f} Gyr"
)

plt.text(1.02, 0.98, info, transform=plt.gca().transAxes,
         verticalalignment='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='paleturquoise', alpha=0.6))

plt.legend(bbox_to_anchor=(1, 0.7))
plt.show()


fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for particle in range(ang_mom_all.shape[0]):
    ax[0].plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, (kinetic_energy_all[particle] + potential_energy_all[particle]), label='Total Energy')
ax[0].set_xlabel('Time [Gyr]')
ax[0].set_ylabel('Total Energy [J]')
ax[0].set_title('Total Energy over Time')




for particle in range(ang_mom_all.shape[0]):
    ax[1].plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, ang_mom_all[particle], label='Total angular momentum')
ax[1].set_xlabel('Time [Gyr]')
ax[1].set_ylabel('Total angular momentum [kg m^2/s]')
ax[1].set_title('Total angular momentum over Time')


plt.tight_layout()
plt.show()




# 2) Static with leapfrog - 1000 timesteps

In [ ]:

import importlib
importlib.reload(ATG)


animate = False
static = True
frozen = False
SphHT = False
integrator = 'leapfrog'

frozen_test_analytic = ATG.StellarSimTDep(m22 = 1, r_half = 0.19, no_of_particles = 10, no_time_steps = 1000, total_evolve_time = 5, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, static = static, frozen = frozen, SphHT = SphHT, integrator = integrator, animate=animate, animate_every=10)

In [ ]:

frozen_test_analytic.run_simulation()

In [ ]:
# Convert to numpy for plotting/analysis
import numpy as np

positions_all = np.array([p.positions_xyz for p in frozen_test_analytic.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in frozen_test_analytic.particles])  # (N_particles, N_steps+1)
v_disp_all    = np.array([p.stellar_v_disp for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1)
kinetic_energy_all = np.array([p.kinetic_energy for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1, 3)

In [ ]:

print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
if animate == True:
    ani = frozen_test_analytic.animator.create_animation()


In [ ]:
time_step2 = frozen_test_analytic.time_step
stellar_v_disp2 = np.mean(v_disp_all, axis=0)  # Average over particles

average_r2 = np.mean(r_all, axis=0)  # Average over particles


x = np.linspace(0, time_step2, len(stellar_v_disp2))

plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, stellar_v_disp2 * frozen_test_analytic.u.to_kms)
plt.yscale('log')
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, average_r2 * frozen_test_analytic.u.to_Kpc, label='Average Particle Radius')
for particle in range(r_all.shape[0]):
    plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, r_all[particle] * frozen_test_analytic.u.to_Kpc, alpha = 0.1, color='gray')
plt.axhline(frozen_test_analytic.r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')
plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')
plt.legend()
plt.show()


fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for particle in range(ang_mom_all.shape[0]):
    ax[0].plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, (kinetic_energy_all[particle] + potential_energy_all[particle]), label='Total Energy')
ax[0].set_xlabel('Time [Gyr]')
ax[0].set_ylabel('Total Energy [J]')
ax[0].set_title('Total Energy over Time')


for particle in range(ang_mom_all.shape[0]):
    ax[1].plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, ang_mom_all[particle], label='Total angular momentum')
ax[1].set_xlabel('Time [Gyr]')
ax[1].set_ylabel('Total angular momentum [kg m^2/s]')
ax[1].set_title('Total angular momentum over Time')


plt.tight_layout()
plt.show()




# 3) Frozen ULDM with ias15 - 100 timesteps

In [ ]:

import importlib
importlib.reload(ATG)


animate = False
static = False
frozen = True
SphHT = False
integrator = 'ias15'

frozen_test_analytic = ATG.StellarSimTDep(m22 = 1, r_half = 0.19, no_of_particles = 10, no_time_steps = 50, total_evolve_time = 5, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, static = static, frozen = frozen, SphHT = SphHT, integrator = integrator, animate=animate, animate_every=10)

In [ ]:

frozen_test_analytic.run_simulation()

In [ ]:
# Convert to numpy for plotting/analysis
import numpy as np

positions_all = np.array([p.positions_xyz for p in frozen_test_analytic.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in frozen_test_analytic.particles])  # (N_particles, N_steps+1)
v_disp_all    = np.array([p.stellar_v_disp for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1)
kinetic_energy_all = np.array([p.kinetic_energy for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1, 3)

In [ ]:

print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
if animate == True:
    ani = frozen_test_analytic.animator.create_animation()


In [ ]:
time_step2 = frozen_test_analytic.time_step
stellar_v_disp2 = np.mean(v_disp_all, axis=0)  # Average over particles

average_r2 = np.mean(r_all, axis=0)  # Average over particles


x = np.linspace(0, time_step2, len(stellar_v_disp2))

plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, stellar_v_disp2 * frozen_test_analytic.u.to_kms)
plt.yscale('log')
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, average_r2 * frozen_test_analytic.u.to_Kpc, label='Average Particle Radius')
for particle in range(r_all.shape[0]):
    plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, r_all[particle] * frozen_test_analytic.u.to_Kpc, alpha = 0.1, color='gray')
plt.axhline(frozen_test_analytic.r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')
plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')
plt.legend()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for particle in range(ang_mom_all.shape[0]):
    ax[0].plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, (kinetic_energy_all[particle] + potential_energy_all[particle]), label='Total Energy')
ax[0].set_xlabel('Time [Gyr]')
ax[0].set_ylabel('Total Energy [J]')
ax[0].set_title('Total Energy over Time')




for particle in range(ang_mom_all.shape[0]):
    ax[1].plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, ang_mom_all[particle], label='Total angular momentum')
ax[1].set_xlabel('Time [Gyr]')
ax[1].set_ylabel('Total angular momentum [kg m^2/s]')
ax[1].set_title('Total angular momentum over Time')


plt.tight_layout()
plt.show()




# 4) Frozen ULDM with leapfrog - 1000 timesteps

In [ ]:

import importlib
importlib.reload(ATG)


animate = False
static = False
frozen = True
SphHT = False
integrator = 'leapfrog'

frozen_test_analytic = ATG.StellarSimTDep(m22 = 1, r_half = 0.19, no_of_particles = 10, no_time_steps = 1000, total_evolve_time = 5, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, static = static, frozen = frozen, SphHT = SphHT, integrator = integrator, animate=animate, animate_every=10)

In [ ]:

frozen_test_analytic.run_simulation()

In [ ]:
# Convert to numpy for plotting/analysis
import numpy as np

positions_all = np.array([p.positions_xyz for p in frozen_test_analytic.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in frozen_test_analytic.particles])  # (N_particles, N_steps+1)
v_disp_all    = np.array([p.stellar_v_disp for p in frozen_test_analytic.particles]) # (N_particles, N_steps+1)


In [ ]:

print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
if animate == True:
    ani = frozen_test_analytic.animator.create_animation()


In [ ]:
time_step2 = frozen_test_analytic.time_step
stellar_v_disp2 = np.mean(v_disp_all, axis=0)  # Average over particles

average_r2 = np.mean(r_all, axis=0)  # Average over particles


x = np.linspace(0, time_step2, len(stellar_v_disp2))

plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, stellar_v_disp2 * frozen_test_analytic.u.to_kms)
plt.yscale('log')
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, average_r2 * frozen_test_analytic.u.to_Kpc, label='Average Particle Radius')
for particle in range(r_all.shape[0]):
    plt.plot(x * frozen_test_analytic.dt * frozen_test_analytic.u.to_Gyr, r_all[particle] * frozen_test_analytic.u.to_Kpc, alpha = 0.1, color='gray')
plt.axhline(frozen_test_analytic.r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')
plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')
plt.legend()
plt.show()




# Testing the different methods of obtaining acceleration from density field

## 1) Static cNFWt Profile using Bessel

In [ ]:

import importlib
importlib.reload(BT)


animate = False
static = True
frozen = False

cNFWt_test = BT.StellarSimTDep(m22 = 1, r_half = 0.19, no_of_particles = 1, no_time_steps = 1000, total_evolve_time = 5, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, N_max = 1000, static = static, frozen = frozen, animate=animate, animate_every=10)

In [ ]:
cNFWt_test.run_simulation()

In [ ]:
pos = np.array(cNFWt_test.positions_xyz)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
ax.plot(pos[:, 0], pos[:, 1], pos[:, 2])
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
ax.plot(pos[:, 0], pos[:, 1], pos[:, 2])
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
ax.plot(pos[:, 0], pos[:, 1], pos[:, 2])
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
if animate == True:
    ani = cNFWt_test.animator.create_animation()


In [ ]:

velocities = np.array(cNFWt_test.velocities)
stellar_v_disp = np.array(cNFWt_test.stellar_v_disp) 
average_r = np.array(cNFWt_test.average_r)
r_values = np.array(cNFWt_test.r_values)
time_step = cNFWt_test.time_step

In [ ]:


x = np.linspace(0, time_step, len(stellar_v_disp))

plt.plot(x * cNFWt_test.dt * cNFWt_test.u.to_Gyr, stellar_v_disp * cNFWt_test.u.to_kms)
plt.yscale('log')
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

plt.plot(x * cNFWt_test.dt * cNFWt_test.u.to_Gyr, average_r * cNFWt_test.u.to_Kpc, label='Average Particle Radius')
#plt.plot(x * first_sim.dt * first_sim.u.to_Gyr, r_values * first_sim.u.to_Kpc, label='Particle radial position')
plt.axhline(cNFWt_test.r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')
plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')
plt.legend()
plt.show()